# Python C extension generator

Use an LLM model to generate a high performance Python C extension code from Python code.

Python C extension modules allows to integrate C coded and compiled modules into Python applications.

* [Python C Extensions](https://docs.python.org/3.13/extending/index.html)
* [Python's C API](https://docs.python.org/3.13/c-api/index.html)

In [ ]:
# Imports.

import os
import sys
from time import perf_counter
from timeit import timeit

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

In [ ]:
# Load environment variables from '.env' file.

load_dotenv(override=True)

In [ ]:
# Initialize client and set the default LLM model to use.

OPENAI_MODEL = "gpt-5.1-codex-mini"

openai = OpenAI()

In [ ]:
# Define Pydantic model class for GPT response parsing.

class Extension_codes(BaseModel):
    """Pydantic model of a response containing the generated C code, the 'setup.py' code and an usage example."""
    c_code: str
    setup: str
    usage: str

In [ ]:
# Define a function to print the optimization codes.

def print_optimization(optimization):
    """Print the optimization codes."""
    print(f"C CODE:\n{optimization.c_code}")
    print("---------------------------")
    print(f"setup.py:\n{optimization.setup}")
    print("---------------------------")
    print(f"USAGE:\n{optimization.usage}")

In [ ]:
# Define a function to write outputs to a file with a given filename.

def write_file(data, filename):
    """Write data to a file with the specified filename."""
    with open(filename, "w") as file:
        file.write(data)

In [ ]:
# Define a function to write the optimization codes to files.

def write_optimization(optimization, module_name):
    """Write the optimization codes to files."""
    write_file(optimization.c_code, f"{module_name}.c")
    write_file(optimization.setup, "setup.py")
    write_file(optimization.usage, "usage_example.py")

In [ ]:
# Define system message for the LLM with instructions for generating the C extension code.

system_message = """
You are an assistant that reimplements Python code in high performance C extensions for Python.
Your responses must always be a JSON with the following structure:

{
    "c_code": "Optimized C extension for Python code",
    "setup": "The 'setup.py' code to compile the C extension for Python",
    "usage": "An example of usage of the C extension for Python code with time measurement and comparing with the original Python code"
}

Use comments sparingly and do not provide any explanation other than occasional comments.
The C extension for Python needs to produce an identical output in the fastest possible time.
Make sure the C extension for Python code is correct and can be compiled with 'python setup.py build' and used in Python.
The usage example must include a time measurement and a comparison with the original Python code.
Do not include any additional text or explanation outside the JSON structure.
Make sure the JSON is correctly formatted.
"""

In [ ]:
# Define user prompt template and function to fill it.

def user_prompt_for(python_code, module_name):
    user_prompt = f"""
    Reimplement this Python code as a C extension for Python with the fastest possible implementation that produces identical output in the least time.
    Respond only with C extension for Python code, do not explain your work other than a few code comments.
    The module name, used to import, must be "{module_name}", the generated C file will be named "{module_name}.c".
    Pay attention to number types to ensure no int overflows.
    Remember to #include all necessary C packages such as iomanip or <python.h>

    The target architecture is {sys.platform}, take that in mind while generating the C code, specially
    when choosing types to use, and use the appropriate compiler flags.
    Make sure to use the Python C API correctly and manage memory properly to avoid leaks or crashes.

    Here is the Python code to reimplement:

    {python_code}"""
    return user_prompt

In [ ]:
# Define function to create the messages for the LLM.

def messages_for(python_code, module_name):
    """Create the messages for the LLM given the Python code and the desired module name."""
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(python_code, module_name)}]

In [ ]:
# Test the messages function and print the messages.

for message in messages_for("print('Hello World')", "say_hello"):
    print(f"{message['role'].upper()}: {message['content']}")
    print("--------------------------------")

In [ ]:
# Define optimization function using OpenAI's GPT model.

def optimize_gpt(python_code, module_name, model=OPENAI_MODEL):
    """Optimize the given Python code by generating a C extension for Python with the specified module name using the specified LLM model."""
    response = openai.responses.parse(
        model=model,
        input=messages_for(python_code, module_name),
        text_format=Extension_codes).output_parsed
    return response

# Try it with a math function that calculates ***π*** using the Leibniz formula.

This formula implies the iterative approximation of *π* using an alternating series,
the more iterations the more the precision but with a cost of more computation.
* [Leibniz formula for π](https://en.wikipedia.org/wiki/Leibniz_formula_for_%CF%80)

This is a good candidate to get a noticeable improvement by coding and compiling it into a Python C extension. 

> NOTE:
>
> We are creating an importable module not an executable program so the code to be optimized must contain only declarations such as DEF or CLASS.

In [ ]:
# Define the Python function to be converted to a C extension and its module name.

module_name = "calculate_pi"

calculate_pi_code = f"""
def leibniz_pi(iterations):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * 4 - 1
        result -= (1/j)
        j = i * 4 + 1
        result += (1/j)
    return result * 4
"""

# Define a function to test the performance of the calculus function.

def test_pi_calculation(calculus_function ,iterations=100_000_000):
    """Test the performance of the given calculus function."""
    start_time = perf_counter()
    result = calculus_function(iterations)
    end_time = perf_counter()
    print(f"Result: {result:.12f}")
    print(f"Execution Time: {(end_time - start_time):.6f} seconds")

# Execute function declaration.
exec(calculate_pi_code)

In [ ]:
# Run original python code and time it.

test_pi_calculation(leibniz_pi, 100_000_000)

In [ ]:
# Average timing the original Python code running it several times.
# (Increase 'iterations' for better timing)

print("Timing...")
iterations = 5
average = timeit(lambda: leibniz_pi(100_000_000), number=iterations) / iterations
print(f"Python average execution time: {average:.6f} seconds")

In [ ]:
# Request code optimization using GPT.

optimization = optimize_gpt(calculate_pi_code, module_name)

In [ ]:
# Print generated extension code.

print_optimization(optimization)

In [ ]:
# Write the generated code to files.
# (Will overwrite existing files)

write_optimization(optimization, module_name)

# Compiling C Extension and executing

The python setup command may fail inside Jupyter lab, if that's the case try it directly on the command line.

There are two cells with WINDOWS ONLY, those are to manage the fact windows comes with two command lines,
the old CMD (MS-DOS style) and the new POWERSHELL (Unix style).

It is controlled by the COMSPEC environment variable.\
*(Using this variable is completely innocuous on UNIX systems, they will simply ignore it)*

Most of command lines present here are Unix style but the building one requires CMD so
we switch to CMD before compiling to later restore the preset one.

In [ ]:
# Clean previous builds.
# (Make sure to run this cell before running the compile cell a second time only)
# (May cast errors if no previous build exists)

!rm -r build/

In [ ]:
# [WINDOWS ONLY]
# Set COMSPEC to cmd.exe to avoid issues with some C compilers on Windows.
# (Remember to restore original COMSPEC after compilation and testing)
preset_comspec = os.environ.get("COMSPEC")
os.environ["COMSPEC"] = "C:\\Windows\\System32\\cmd.exe"

In [ ]:
# Compile the C extension.
# (Will fail no C compiler is installed)
# (In case of errors, try directly on the command line)

!python setup.py build_ext --inplace

In [ ]:
# [WINDOWS ONLY]
# Restore original COMSPEC.

os.environ["COMSPEC"] = preset_comspec

In [ ]:
# Run the usage example to test the compiled C extension.
exec(optimization.usage)

In [ ]:
# Import newly created C extension and compare performance with original Python code.

from calculate_pi import leibniz_pi as c_leibniz_pi

print("Testing original Python code:")
test_pi_calculation(leibniz_pi, 100_000_000)
print("Testing C extension code:")
test_pi_calculation(c_leibniz_pi, 100_000_000)
